### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) GROQ 인증키 설정

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [2]:
import langchain
print(langchain.__version__)

1.4.3


In [4]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()
print(f'model name = {llm.model_name}')

model name = openai/gpt-oss-120b


## 2. LLM Chain

### 1) Prompt + LLM


In [5]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델이 어떻게 “배우는지” 쉽게 이해하기\n\n아래는 **머신러닝·딥러닝 모델**이 학습하는 과정을 일상적인 비유와 함께 단계별로 정리한 내용입니다.  \n※ 여기서 말하는 “인공지능 모델”은 사진을 구분하거나, 글을 번역하거나, 음성을 인식하는 등 **데이터를 입력받아 원하는 출력을 만들어내는 프로그램**을 의미합니다.\n\n---\n\n## 1️⃣ “학생”과 “교재”를 만든다  \n| 요소 | 머신러닝에서의 의미 | 비유 |\n|------|-------------------|------|\n| **데이터(교재)** | 모델에게 보여줄 입력‑출력 쌍. 예: 고양이·강아지 사진 + 라벨(‘고양이’, ‘강아지’) | 교과서·문제집 |\n| **모델(학생)** | 입력을 받아서 출력(예측)을 만드는 수학식·네트워크 구조. 예: 신경망, 결정트리 등 | 학생이 문제를 푸는 방법(공부법) |\n| **목표(시험 점수)** | 정답과 얼마나 차이가 나는지 측정하는 **손실 함수(Loss)**. | 시험 점수(오답이 많을수록 점수 낮음) |\n\n---\n\n## 2️⃣ “문제 풀고 답 확인하기” – **손실(Loss) 계산**  \n1. **예측**: 모델에 입력을 주면 현재 학습된 파라미터(가중치)로 **예측값**을 만든다.  \n2. **정답과 비교**: 예측값과 실제 정답(라벨)을 비교해 **오차**를 구한다.  \n3. **손실 함수**: 오차를 숫자로 만든 것이 손실.  \n   - 회귀(숫자 예측) → 평균제곱오차(MSE)  \n   - 분류(카테고리 예측) → 교차엔트로피(Cross‑Entropy)  \n\n> **핵심**: 손실이 작을수록 “학생이 시험을 잘 본” 것과 동일합니다.\n\n---\n\n## 3️⃣ “틀린 문제를 다시 공부한다” – **경사 하강법(Gradient Descent)**  \n손실을 최소화하려면 모델이 **가중치**

In [ ]:
print(result.content)

### 2) PromptTemplate + LLM

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [8]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='### 인공지능 모델이 어떻게 배우는지 쉽게 설명해볼게요  \n\n---\n\n## 1. **학습 데이터가 ‘교재’예요**\n- **입력 (X)** : 사진, 텍스트, 소리 등 모델이 보고 배우는 원본 정보  \n- **정답 (Y)** : “이 사진은 고양이야”, “이 문장은 긍정이야” 같은 정답 라벨  \n\n마치 학생이 교과서와 답안지를 함께 보는 것과 비슷합니다.  \n\n---\n\n## 2. **모델은 ‘학생’이지만 아직은 초보**\n- 모델은 **수많은 파라미터(가중치)** 로 이루어진 수학식이에요.  \n- 처음엔 이 파라미터가 무작위 값이라서 전혀 맞지 않는 답을 내놓아요.  \n\n예를 들어, 고양이와 개를 구분하는 모델이라면 처음엔 “고양이인가요?” 라는 질문에 무작위로 “네” 혹은 “아니오”라고 대답합니다.\n\n---\n\n## 3. **‘손실 함수(Loss)’가 점수표 역할을 해요**\n- 모델이 낸 답과 정답 사이의 차이를 **숫자(손실)** 로 나타냅니다.  \n- 손실이 작을수록 모델이 잘 맞춘 거고, 손실이 클수록 틀린 거예요.  \n\n> **비유** : 시험 점수표. 점수가 낮을수록(손실이 작을수록) 좋은 성적!\n\n---\n\n## 4. **‘경사 하강법(Gradient Descent)’이 공부 방법이에요**\n- 손실이 큰 이유를 찾아서 파라미터를 조금씩 조정합니다.  \n- “내가 이 부분을 더 크게(또는 작게) 만들면 손실이 줄어들까?” 라는 질문을 수천 번, 수백만 번 반복합니다.  \n\n> **비유** : 산을 내려가면서 가장 가파른 길을 찾아 내려가는 것. 손실이 산 꼭대기라면, 가장 가파른 방향(기울기)을 따라 내려가면 낮은 곳(손실이 작아진 곳)에 도착합니다.\n\n---\n\n## 5. **에포크(Epoch

In [9]:
print(result.content)

### 인공지능 모델이 어떻게 배우는지 쉽게 설명해볼게요  

---

## 1. **학습 데이터가 ‘교재’예요**
- **입력 (X)** : 사진, 텍스트, 소리 등 모델이 보고 배우는 원본 정보  
- **정답 (Y)** : “이 사진은 고양이야”, “이 문장은 긍정이야” 같은 정답 라벨  

마치 학생이 교과서와 답안지를 함께 보는 것과 비슷합니다.  

---

## 2. **모델은 ‘학생’이지만 아직은 초보**
- 모델은 **수많은 파라미터(가중치)** 로 이루어진 수학식이에요.  
- 처음엔 이 파라미터가 무작위 값이라서 전혀 맞지 않는 답을 내놓아요.  

예를 들어, 고양이와 개를 구분하는 모델이라면 처음엔 “고양이인가요?” 라는 질문에 무작위로 “네” 혹은 “아니오”라고 대답합니다.

---

## 3. **‘손실 함수(Loss)’가 점수표 역할을 해요**
- 모델이 낸 답과 정답 사이의 차이를 **숫자(손실)** 로 나타냅니다.  
- 손실이 작을수록 모델이 잘 맞춘 거고, 손실이 클수록 틀린 거예요.  

> **비유** : 시험 점수표. 점수가 낮을수록(손실이 작을수록) 좋은 성적!

---

## 4. **‘경사 하강법(Gradient Descent)’이 공부 방법이에요**
- 손실이 큰 이유를 찾아서 파라미터를 조금씩 조정합니다.  
- “내가 이 부분을 더 크게(또는 작게) 만들면 손실이 줄어들까?” 라는 질문을 수천 번, 수백만 번 반복합니다.  

> **비유** : 산을 내려가면서 가장 가파른 길을 찾아 내려가는 것. 손실이 산 꼭대기라면, 가장 가파른 방향(기울기)을 따라 내려가면 낮은 곳(손실이 작아진 곳)에 도착합니다.

---

## 5. **에포크(Epoch)와 배치(Batch)**
- **배치** : 한 번에 보는 데이터 양. 메모리와 연산 효율을 위해 전체 데이터를 작은 조각으로 나눠서 학습합니다.  
- **에포크** : 전체 데이터를 한 번 다 보는 과정. 여러 번 반복하면서 점점 더 좋은 파라미터를 찾습니다. 

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser
* invoke() 함수는 동기(synchronous) 방식으로 호출함

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

### 4) PromptTemplate + LLM(stream()) + StrOutputParser
* stream() 함수는 비동기(asynchronous) 방식으로 호출함

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**영화 추천: “버닝” (Burning, 2018)**  

- **감독**: 이창동  
- **주연**: 이병헌, 전여빈, 스티븐 연  
- **줄거리**: 어린 시절부터 절친했던 두 사람(이병헌·전여빈)이 우연히 만난 미스터리한 청년(스티븐 연)에게 끌리면서, 사랑과 질투, 그리고 ‘소외’라는 감정을 통해 현대 사회의 불안과 인간 내면을 섬세히 파고든 작품입니다.  
- **추천 포인트**  
  1. **깊이 있는 서사** – 일상적인 이야기가 점차 복잡한 심리 스릴러로 전개돼, 관객에게 여러 해석의 여지를 줍니다.  
  2. **연기력** – 이병헌·전여빈·스티븐 연 모두 섬세하고 강렬한 연기를 선보이며, 인물의 감정을 그대로 전달합니다.  
  3. **시각적 미학** – 이창동 감독 특유의 차분하고 아름다운 촬영이 이야기와 어우러져 시청 경험을 풍부하게 합니다.  
  4. **수상 경력** – 칸 영화제 비경쟁 부문 초청 상영, 베니스 국제영화제 심사위원 특별상 등 국제 영화제에서 다수 인정받았습니다.  

> **한 줄 요약**: 일상 속에 숨겨진 불안과 욕망을 섬세히 탐구한, 현대 드라마의 걸작이라 할 수 있는 작품입니다.  

※ 현재 넷플릭스와 일부 OTT(웨이브, 왓챠)에서 스트리밍 중이니 편하게 감상하실 수 있습니다. 즐거운 감상 되세요! 🎬


In [14]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
# prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
#                                             줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001FC2CB7DD00>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001FC168D5C40>, root_client=<openai.OpenAI object at 0x000001FC169461B0>, root_async_client=<openai.AsyncOpenAI object at 0x000001FC2CF1D370>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)

In [17]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")


🔹 1단계: 'action' 장르 영화 추천
   → 추천 영화: "다이 하드 (Die Hard)"는 액션 장르를 대표하는 매우 유명한 영화 중 하나입니다.

 2단계: '"다이 하드 (Die Hard)"는 액션 장르를 대표하는 매우 유명한 영화 중 하나입니다.' 영화 정보 조회
   → 영화 정보:
**다이 하드 (Die Hard, 1988)**  

| 구분 | 내용 |
|------|------|
| **감독** | **존 맥티어넌 (John McTiernan)** – 액션 영화와 스릴러 장르에서 다수의 명작을 만든 미국 감독. |
| **주연 배우** | • **브루스 윌리스 (Bruce Willis)** – 주인공 **존 매클레인 (John McClane)** 형사 <br>• **앨런 립리 (Alan Rickman)** – 악당 **한스 그루버 (Hans Gruber)** <br>• **Bonnie Bedelia** – 매클레인의 아내 **홀리 (Holly)** <br>• **Reginald VelJohnson** – 경찰관 **Al Powell** <br>• **Alexander Godunov** – 테러리스트 **Karl** 등 |
| **개봉** | 1988년 7월 15일 (미국) |
| **제작·배급** | 20th Century Fox (현재는 디즈니/폭스) |
| **장르** | 액션·스릴러·범죄 |

---

### 간단한 줄거리

미국 뉴욕 경찰관 **존 매클레인**은 크리스마스 이브에 LA에 있는 고층 빌딩 **Nakatomi Plaza**(Nakatomi 사옥)로 아내 **홀리**를 만나러 간다.  
그곳에서 Nakatomi 사는 연례 파티가 열리고, 매클레인은 파티에 잠시 끼어들어 **‘테러리스트’**라 자칭하는 **한스 그루버** 일당에게 포위된다.  

그들은 빌딩 안에 있는 1억 달러 규모의 국채를 강탈하려는 목적으로 고도로 조직된 인질극을 벌이며, 경찰과 FBI는 외부에서 상황을 통제하려 한다. 하지만 매클레인은 건물 안에